In [3]:
import os
import cv2
import numpy as np

# -------------------------------------------------------------------------
# 1. Carregamento da Imagem
# -------------------------------------------------------------------------
caminho = "bentoespock.jpeg"
if not os.path.exists(caminho):
    caminho = "bentoespock_3.jpeg"

img_bgr = cv2.imread(caminho)
if img_bgr is None:
    raise FileNotFoundError(f"Não foi possível carregar o arquivo '{caminho}'.")

altura, largura = img_bgr.shape[:2]

# =========================================================================
# PARTE A: Equalização Global vs. CLAHE (no espaço CIE-Lab)
# =========================================================================

# Conversão para o espaço Lab: L (Luminância), a e b (Canais cromáticos)
img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
canal_l, canal_a, canal_b = cv2.split(img_lab)

# 1. Equalização Global do canal de Luminância L
l_eq_global = cv2.equalizeHist(canal_l)
lab_eq_global = cv2.merge([l_eq_global, canal_a, canal_b])
img_eq_global = cv2.cvtColor(lab_eq_global, cv2.COLOR_LAB2BGR)
cv2.imwrite("aula03_01_equalizacao_global.jpeg", img_eq_global)

# 2. Equalização Adaptativa com CLAHE
# clipLimit define o limite de amplificação de contraste; tileGridSize define a vizinhança
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
l_clahe = clahe.apply(canal_l)
lab_clahe = cv2.merge([l_clahe, canal_a, canal_b])
img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)
cv2.imwrite("aula03_02_clahe.jpeg", img_clahe)


# =========================================================================
# PARTE B: Transformação de Perspectiva (Warp Perspective)
# =========================================================================

# Seleção de 4 vértices formando um quadrilátero inclinado ao redor da face branca
# Ordem dos pontos: [Superior Esquerdo, Superior Direito, Inferior Direito, Inferior Esquerdo]
pontos_origem = np.float32([
    [largura * 0.40, altura * 0.18],  # Ponto superior próximo à orelha
    [largura * 0.78, altura * 0.38],  # Ponto superior direito (lateral da cabeça)
    [largura * 0.45, altura * 0.80],  # Ponto inferior direito (base da face/patas)
    [largura * 0.05, altura * 0.55]   # Ponto inferior esquerdo
])

# Dimensões da imagem plana de saída (janela retificada 400x500)
largura_saida = 400
altura_saida = 500

pontos_destino = np.float32([
    [0, 0],
    [largura_saida - 1, 0],
    [largura_saida - 1, altura_saida - 1],
    [0, altura_saida - 1]
])

# Cálculo da Matriz de Homografia 3x3
matriz_perspectiva = cv2.getPerspectiveTransform(pontos_origem, pontos_destino)

# Aplicação da deformação projetiva
img_retificada = cv2.warpPerspective(
    img_bgr, matriz_perspectiva, (largura_saida, altura_saida)
)
cv2.imwrite("aula03_03_warp_perspective.jpeg", img_retificada)

# Marcação visual dos pontos de origem na imagem original para os slides
img_pontos_marcados = img_bgr.copy()
for pt in pontos_origem.astype(int):
    cv2.circle(img_pontos_marcados, tuple(pt), 8, (0, 0, 255), -1)

# Desenha as linhas do polígono projetivo
cv2.polylines(img_pontos_marcados, [pontos_origem.astype(int)], isClosed=True, color=(0, 255, 255), thickness=2)
cv2.imwrite("aula03_04_pontos_origem.jpeg", img_pontos_marcados)

# =========================================================================
# Exibição Interativa
# =========================================================================
cv2.imshow("1. Original", img_bgr)
cv2.imshow("2. Equalizacao Global (Saturada)", img_eq_global)
cv2.imshow("3. CLAHE (Contraste Natural)", img_clahe)
cv2.imshow("4. Poligono de Perspectiva", img_pontos_marcados)
cv2.imshow("5. Face Retificada (Warp Perspective)", img_retificada)

print("Imagens salvas. Pressione qualquer tecla para encerrar...")
cv2.waitKey(0)
cv2.destroyAllWindows()

Imagens salvas. Pressione qualquer tecla para encerrar...
